# Topic Modelling the Voynich Manuscript (NMF, LDA, BERTopic)

## 1. Environment
- **Platform**: Kaggle Notebooks  
- **Accelerator**: GPU (recommended for BERTopic)  
  - In the notebook editor: **Settings → Accelerator → GPU**

## 2. Dataset
We use the **Voynich by quimqu** dataset, available on Kaggle.  

Add it to your notebook via:  
- **Add data → Search → “Voynich by quimqu” → Add to notebook**

This mounts the dataset under:  
```
/kaggle/input/voynich-by-quimqu/
```

Load the main file:

```python
import pandas as pd
df = pd.read_csv("/kaggle/input/voynich/voynich_paragraphs.csv")
```

It contains columns like:
```
tag | line | paragraph | quire | page | folio | bifolio | section | language | ...
```


## 3. Outputs
- PNGs under `/kaggle/working/plots/...`
- GIFs (animations) under `/kaggle/working/plots_gifs/...`

---

⚡ **Tip**: BERTopic will automatically use the GPU (through `sentence-transformers`) if GPU is enabled in Kaggle.


In [ ]:
!pip install -q bertopic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.0/153.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 96.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 76.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 30.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 79.0 MB/s eta 0:00:00:00:0100:01


In [ ]:
# --- Core Python ---
import os
import re
import glob
import json
from typing import List, Tuple, Dict, Optional

# --- Numerical & Data Analysis ---
import numpy as np
import pandas as pd

# --- Visualization ---
import matplotlib.pyplot as plt
from PIL import Image

# --- Machine Learning / NLP ---
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.decomposition import NMF as SKNMF
from sklearn.preprocessing import normalize
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import issparse
from bertopic import BERTopic


In [ ]:
# Initial variables

text_to_read = "voynich"   # by now, "voynich", "timm2", "timm3", "timm4"
test_best_k = False        # True calculates optimum number of topics
random = False             # True shuffles the text
filter_section = None      # Write section to filter dataset "Astronomical", "Biological (balneological)", "Cosmological"
                           # "Herbal", "Pharmaceutical", "Marginal stars only", "Text-only", "Zodiac"
min_K = 2                  # minimum topic quantity to be found
max_K = 10                 # maximum topic quantity to be found
plot_by_folio = True       # plot results grouped by folio and section
plot_by_paragraph = False  # plot results by paragraph and section
group_paragraphs = True    # group paragraphs as a single unit. If False, each line is a single unit.
plot_group_by = "section"  # can be "section", "folio", "language", "w_hand", "c_hand"

In [ ]:
# Load the DataFrame
df = pd.read_csv("/kaggle/input/voynich/voynich_paragraphs.csv")

# Load the mapping
with open("/kaggle/input/voynich/voynich_mapping.json", "r", encoding="utf-8") as f:
    mapping = json.load(f)

special_to_unicode = mapping["special_to_unicode"]
unicode_to_special = mapping["unicode_to_special"]

print("Sample mapping:")
print(list(special_to_unicode.items())[:5])  # first 5 entries

df.head()


In [ ]:
if filter_section is not None:

    print("Original rows:", len(df))
    
    # Keep only rows where section == "Marginal stars only"
    df = df[df["section"] == filter_section].copy()
    
    print("Filtered rows:", len(df))
    df.head()


In [ ]:
# 1) Load words from a Timm file (ignores comment lines starting with '#')
def load_timm_words(path):
    """
    Reads a text file containing words from a Timm corpus.
    - Skips empty lines and lines starting with '#'
    - Splits each valid line into words separated by whitespace
    - Returns a flat list of words
    """
    words = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.lstrip().startswith("#"):
                continue  # skip comments and empty lines
            words.extend(s.split())  # split by spaces and add to list
    return words


# 2) Replace df['paragraph'] content with words from an external list (e.g., Timm2),
#    while keeping the original word count per paragraph.
def replace_paragraph_words_from_list(
    df,
    src_words,
    col="paragraph",
    in_place=True,
    on_short="error"
):
    """
    Replace each paragraph in a DataFrame with new words taken sequentially
    from a source word list, preserving the same number of words per paragraph.

    Parameters
    ----------
    df : pd.DataFrame
        Input DataFrame containing a text column (default: 'paragraph').
    src_words : list[str]
        Source list of words that will replace the original text.
    col : str
        Column name in df to modify.
    in_place : bool
        If True, modifies df directly. If False, returns a list of new texts.
    on_short : {'error', 'cycle', 'truncate'}
        What to do if src_words has fewer words than needed:
        - 'error'    : raise an exception (default).
        - 'cycle'    : repeat src_words until enough words are available.
        - 'truncate' : use what is available and leave later paragraphs shorter/empty.

    Returns
    -------
    pd.DataFrame or list[str]
    """
    # Count number of words needed for each paragraph
    counts = df[col].fillna("").map(lambda s: len(s.split()))
    need = int(counts.sum())     # total words needed
    have = len(src_words)        # total words available

    # Handle shortage of words
    if have < need:
        if on_short == "error":
            raise ValueError(
                f"Not enough words in source list: need {need}, have {have}. "
                f"Use on_short='cycle' or 'truncate' to override."
            )
        elif on_short == "cycle":
            # Repeat the source list until we cover 'need'
            times = (need + have - 1) // have
            src_words = (src_words * times)[:need]
        elif on_short == "truncate":
            # Just use what is available; some paragraphs may be incomplete
            pass
        else:
            raise ValueError("on_short must be 'error', 'cycle', or 'truncate'.")

    # Build new paragraphs with the same word counts
    out = []
    pos = 0
    for n in counts:
        n = int(n)
        if pos + n <= len(src_words):
            chunk = src_words[pos:pos+n]
        else:
            chunk = src_words[pos:]  # may be empty if words run out
        pos += n
        out.append(" ".join(chunk))

    if in_place:
        # Optionally save the original column before overwriting:
        # df[col + "_orig"] = df[col]
        df[col] = out
        return df
    else:
        return out


# 3) Shuffle the entire corpus and reassign words to paragraphs,
#    preserving the original paragraph lengths.
def shuffle_corpus_and_reassign(
    df: pd.DataFrame,
    col: str = "paragraph",
    seed: int | None = None,
    in_place: bool = True
):
    """
    Shuffle all words in a text column and redistribute them back into paragraphs
    with the same lengths as the originals.

    Parameters
    ----------
    df : pd.DataFrame
        Input DataFrame with a text column (default: 'paragraph').
    col : str
        Column name containing the text to shuffle.
    seed : int or None
        Random seed for reproducibility.
    in_place : bool
        If True, modifies df directly. If False, returns a list of new texts.

    Returns
    -------
    pd.DataFrame or list[str]
    """
    # 1) Tokenize each paragraph into words (split on whitespace)
    paragraphs = df[col].fillna("").astype(str).tolist()
    tokens_per_row = [p.split() for p in paragraphs]
    lengths = [len(toks) for toks in tokens_per_row]  # keep paragraph lengths

    # 2) Flatten the corpus and shuffle all words
    words = [w for toks in tokens_per_row for w in toks]
    rng = np.random.default_rng(seed)
    perm = rng.permutation(len(words))
    shuffled = [words[i] for i in perm]

    # 3) Reassign words to paragraphs, preserving original lengths
    out_texts = []
    pos = 0
    for n in lengths:
        out_texts.append(" ".join(shuffled[pos:pos+n]))
        pos += n

    # 4) Return or overwrite DataFrame
    if in_place:
        # df[col + "_orig"] = df[col]   # uncomment if you want to keep original
        df[col] = out_texts
        return df
    else:
        return out_texts


# --- Example usage ---
if "timm" in text_to_read:
    # Load Timm words from external file
    path = "/kaggle/input/voynich/" + text_to_read + ".txt"
    timm_words = load_timm_words(path)
    # Replace paragraphs with these words, keeping same lengths
    replace_paragraph_words_from_list(df, timm_words, col="paragraph", in_place=True, on_short="error")

if random:
    # Randomly shuffle entire corpus and redistribute words to paragraphs
    shuffle_corpus_and_reassign(df, col="paragraph", seed=42, in_place=True)


In [ ]:
def group_short_paragraphs(
    df: pd.DataFrame,
    text_col: str = "paragraph",
    group_col: str = "tag",
    max_words: int = 4,
    sep: str = " "
):
    """
    Group and merge very short paragraphs within each group.

    For each group defined by `group_col` (e.g., 'tag'):
      - All rows where `text_col` has <= max_words words are merged into a single row.
      - The merged text is a concatenation of those short paragraphs, joined with `sep`.
      - Metadata columns (all other columns) are taken from the FIRST short paragraph
        encountered in that group.
      - Paragraphs with > max_words words are left unchanged.

    Parameters
    ----------
    df : pd.DataFrame
        Input DataFrame containing a column of text paragraphs.
    text_col : str
        Name of the column containing paragraph text.
    group_col : str
        Column used to define groups (e.g. 'tag' = folio ID).
    max_words : int
        Maximum number of words for a paragraph to be considered "short".
    sep : str
        Separator used when concatenating short paragraphs.

    Returns
    -------
    pd.DataFrame
        A DataFrame where short paragraphs within each group have been merged,
        preserving the original order of rows as much as possible.
    """
    df = df.copy()
    # Track the original order so we can restore it after transformations
    df["__order"] = np.arange(len(df))

    # Count words in each paragraph (split by whitespace)
    wc = df[text_col].fillna("").str.split().str.len()
    short = wc <= max_words

    # If there are no short paragraphs, return df unchanged (except drop helper col)
    if not short.any():
        return df.drop(columns="__order")

    # Aggregate all short paragraphs per group:
    # - Concatenate their texts
    # - Track the earliest position (min __order) to keep group order consistent
    agg = (
        df[short]
        .groupby(group_col, as_index=False)
        .agg(
            **{
                text_col: (text_col, lambda s: sep.join(s.tolist()).strip()),
                "__order": ("__order", "min"),
            }
        )
    )

    # Identify the first short row per group (to copy its metadata)
    first_short_rows = (
        df[short]
        .sort_values([group_col, "__order"])
        .groupby(group_col, as_index=False)
        .first()
    )

    # Replace the text of the first short row with the aggregated short text,
    # while keeping all the other metadata intact
    reps = (
        first_short_rows.drop(columns=[text_col])
        .merge(agg[[group_col, text_col]], on=group_col, how="left")
    )

    # Remove all original short rows from the DataFrame
    remaining = df[~short]

    # Combine the remaining rows with the replacements,
    # then sort back into the original group/order sequence
    out = (
        pd.concat([remaining, reps], ignore_index=True, sort=False)
        .sort_values([group_col, "__order"])
        .drop(columns="__order")
        .reset_index(drop=True)
    )
    return out


if group_paragraphs:

    # Apply the function and then sort by another column (e.g., 'order0')
    df2 = group_short_paragraphs(df).sort_values(by="order0")
    
    # Pick one tag as an example (could also specify explicitly, e.g., 'f1r')
    tag_example = df['tag'].iloc[0]

else:

    df2 = df.copy()

print("Before:", len(df), "paragraphs")
print("After:", len(df2), "paragraphs")
print("Example paragraph:", df2.iloc[5]['paragraph'])


In [ ]:
# ---------- Utilitiess ----------
def hoyer_sparsity(M, axis=1, eps=1e-12):
    """
    Compute Hoyer's sparsity for each vector (row-wise if axis=1, column-wise if axis=0)
    and return the average sparsity.

    Hoyer sparsity ∈ [0, 1], where:
      - 0 means fully dense (uniform distribution),
      - 1 means maximally sparse (all mass on a single coordinate).

    Parameters
    ----------
    M : array-like or sparse matrix
        Matrix whose sparsity you want to measure.
    axis : int
        1 to compute sparsity per row, 0 per column.
    eps : float
        Small constant to avoid division by zero.

    Returns
    -------
    float
        Average Hoyer sparsity across the chosen axis.
    """
    if issparse(M):
        M = M.toarray()
    if axis == 0:
        M = M.T
    n = M.shape[1]
    l1 = np.linalg.norm(M, 1, axis=1)
    l2 = np.linalg.norm(M, 2, axis=1)
    s = (np.sqrt(n) - (l1 + eps) / (l2 + eps)) / (np.sqrt(n) - 1)
    s = np.clip(s, 0, 1)
    return float(np.nanmean(s))

def avg_topic_similarity(H):
    """
    Compute the average cosine similarity between topics (rows of H),
    excluding the diagonal. Lower values indicate more dissimilar (better separated) topics.

    Parameters
    ----------
    H : array-like, shape (K, V)
        Topic-term matrix where each row is a topic distribution over the vocabulary.

    Returns
    -------
    float
        Mean pairwise cosine similarity across all topic pairs.
    """
    S = cosine_similarity(H)
    k = S.shape[0]
    return float((S.sum() - np.trace(S)) / (k * (k - 1)))

def topic_stability(H1, H2):
    """
    Greedy 1–1 matching of topics between two runs using cosine similarity.
    Works when the two runs have different numbers of topics (rectangular S).
    Returns the mean matched similarity over min(n_topics1, n_topics2).
    """
    # Ensure dense 2-D
    if hasattr(H1, "toarray"): H1 = H1.toarray()
    if hasattr(H2, "toarray"): H2 = H2.toarray()
    H1 = np.asarray(H1)
    H2 = np.asarray(H2)
    if H1.ndim == 1: H1 = H1.reshape(1, -1)
    if H2.ndim == 1: H2 = H2.reshape(1, -1)

    # Cosine similarity between topic rows
    S = cosine_similarity(H1, H2)   # shape (r, c)
    r, c = S.shape
    m = min(r, c)
    if m == 0:
        return np.nan

    used_r, used_c = set(), set()
    total = 0.0
    for _ in range(m):
        best_i = best_j = -1
        best_val = -np.inf
        for i in range(r):
            if i in used_r: 
                continue
            for j in range(c):
                if j in used_c:
                    continue
                val = S[i, j]
                if val > best_val:
                    best_val = val
                    best_i, best_j = i, j
        if best_i < 0 or best_j < 0:
            break
        used_r.add(best_i)
        used_c.add(best_j)
        total += best_val

    return float(total / max(1, len(used_r)))
    
def umass_coherence(H, X_bin, top_n=15, eps=1.0):
    """
    UMass coherence on top_n terms per topic.
    Robust to sparse inputs and 1-D rows; clips top_n to n_terms.
    """
    # --- ensure H is dense 2-D (n_topics, n_terms) ---
    if hasattr(H, "toarray"):
        H = H.toarray()
    H = np.asarray(H)
    if H.ndim == 1:
        H = H.reshape(1, -1)

    n_topics, n_terms = H.shape
    if n_terms == 0 or n_topics == 0:
        return np.nan

    # clip top_n to available terms
    top_n = int(max(1, min(top_n, n_terms)))

    # --- ensure X_bin has matching vocab dimension and is dense int ---
    if hasattr(X_bin, "toarray"):
        Xb = X_bin.toarray()
    else:
        Xb = np.asarray(X_bin)
    if Xb.ndim != 2 or Xb.shape[1] != n_terms:
        raise ValueError(f"X_bin shape {Xb.shape} incompatible with H shape {H.shape}")

    Xb = (Xb > 0).astype(int)

    # indices of top words per topic
    top_idx_per_topic = np.argpartition(H, -top_n, axis=1)[:, -top_n:]
    unique_idx = np.unique(top_idx_per_topic.ravel())

    # restrict to sub-vocabulary for efficiency
    Xb_sub = Xb[:, unique_idx]                    # (D x T_sub)
    C = (Xb_sub.T @ Xb_sub).astype(float)         # (T_sub x T_sub) co-occurrence
    Dw = C.diagonal().copy()                      # docs with w
    # map original term -> sub-vocab position
    pos_in_sub = {idx: p for p, idx in enumerate(unique_idx)}

    scores = []
    for t in range(n_topics):
        top = top_idx_per_topic[t]
        # order top terms by descending weight
        top_sorted = top[np.argsort(H[t, top])][::-1]
        top_sub = [pos_in_sub[i] for i in top_sorted]
        vals = []
        for i in range(1, len(top_sub)):
            wi = top_sub[i]
            for j in range(i):
                wj = top_sub[j]
                num = C[wi, wj] + eps
                den = Dw[wj] + eps
                vals.append(np.log(num / den))
        if vals:
            scores.append(np.mean(vals))
    return float(np.mean(scores)) if scores else np.nan

def fit_nmf(X, k, random_state=42, max_iter=500):
    """
    Fit a single NMF model with common, stable hyperparameters for text topic modeling.

    Parameters
    ----------
    X : sparse or ndarray, shape (D, V)
        TF-IDF matrix (documents x terms).
    k : int
        Number of topics.
    random_state : int
        Seed for reproducibility.
    max_iter : int
        Maximum number of optimization iterations.

    Returns
    -------
    nmf : fitted sklearn.decomposition.NMF
    W : ndarray, shape (D, K)
        Document-topic matrix.
    H : ndarray, shape (K, V)
        Topic-term matrix.
    """
    nmf = SKNMF(
        n_components=k,
        init="nndsvda",
        random_state=random_state,
        max_iter=max_iter,
        solver="cd",
        beta_loss="frobenius",
        tol=1e-4,
    )
    W = nmf.fit_transform(X)
    H = nmf.components_
    return nmf, W, H

# ---------- Main pipeline ----------
def evaluate_nmf_k(
    paragraphs,
    k_values=range(2, 31),
    ngram_range=(1, 1),
    min_df=2,
    max_df=0.95,
    max_features=None
):
    """
    Evaluate NMF topic models across multiple K values and visualize metrics.

    For each K, we:
      - Fit NMF on a TF-IDF representation of the input paragraphs.
      - Compute several diagnostics:
          * reconstruction_error: raw Frobenius reconstruction error
          * relative_error: reconstruction_error normalized by ||X||
          * explained_variance_like: 1 - (reconstruction_error / ||X||)^2
          * avg_topic_similarity: mean cosine similarity between topics (lower is better)
          * umass_coherence: topic coherence using a binary DTM (higher is better)
          * silhouette_on_W: clustering quality using hard assignments on W (may be NaN if degenerate)
          * sparsity_W / sparsity_H: Hoyer sparsity of W and H (higher is sparser)
          * stability: agreement between two NMF runs with different seeds

    Parameters
    ----------
    paragraphs : iterable of str
        Texts to model (e.g., df2['paragraph']).
    k_values : iterable of int
        Range of topic counts to evaluate.
    ngram_range, min_df, max_df, max_features :
        TF-IDF vectorizer configuration. No stopword removal here since the
        Voynich script often relies on exact surface forms.
    """
    # TF-IDF (word-level tokenization). We keep case and accents as-is for Voynich.
    vectorizer = TfidfVectorizer(
        ngram_range=ngram_range,
        min_df=min_df,
        max_df=max_df,
        max_features=max_features,
        strip_accents=None,
        lowercase=False,   # Voynich may be case/diacritic-sensitive
        token_pattern=r"(?u)\b\w+\b"  # alphanumeric "words"; adjust if you have a custom tokenizer
    )
    X = vectorizer.fit_transform(paragraphs)

    # Norm of X used to normalize reconstruction error across K
    normX = np.linalg.norm(X.data) if issparse(X) else np.linalg.norm(X)

    # Binary DTM for UMass coherence (presence/absence)
    X_bin = (X > 0).astype(int)

    rows = []
    for k in k_values:
        # Main model
        nmf1, W1, H1 = fit_nmf(X, k, random_state=0)

        # Errors and a simple "explained variance-like" score
        frob = nmf1.reconstruction_err_
        rel_err = frob / (normX + 1e-12)
        expl_like = 1.0 - (frob / (normX + 1e-12))**2

        # Topic dissimilarity (lower is better) using cosine on L1-normalized topics
        H1n = normalize(H1, norm="l1", axis=1)
        avg_sim = avg_topic_similarity(H1n)

        # Topic coherence (higher is better; often negative)
        coh_umass = umass_coherence(H1n, X_bin, top_n=15)

        # Silhouette over document-topic distributions (argmax hard labels)
        W1n = normalize(W1, norm="l1", axis=1)
        labels = np.argmax(W1n, axis=1)
        try:
            sil = silhouette_score(W1n, labels, metric="cosine")
        except Exception:
            # E.g., if a cluster ends up with only 1 document, silhouette is undefined
            sil = np.nan

        # Sparsity of document-topic and topic-term matrices
        sp_W = hoyer_sparsity(W1n, axis=1)
        sp_H = hoyer_sparsity(H1n, axis=1)

        # Stability: fit a second model with a different seed and compare topics
        _, _, H2 = fit_nmf(X, k, random_state=42)
        H2n = normalize(H2, norm="l1", axis=1)
        stab = topic_stability(H1n, H2n)

        rows.append({
            "K": k,
            "reconstruction_error": frob,
            "relative_error": rel_err,
            "explained_variance_like": expl_like,
            "avg_topic_similarity": avg_sim,
            "umass_coherence": coh_umass,
            "silhouette_on_W": sil,
            "sparsity_W": sp_W,
            "sparsity_H": sp_H,
            "stability": stab,
        })

    metrics = pd.DataFrame(rows).set_index("K")

    # -------- Plots (one per metric) --------
    def plot_metric(df, col, ylabel=None):
        plt.figure(figsize=(7, 4))
        df[col].plot(marker="o")
        plt.xlabel("K")
        plt.ylabel(ylabel or col)
        plt.title(col)
        plt.grid(True, alpha=0.3)
        plt.show()

    for col in [
        "reconstruction_error",
        "relative_error",
        "explained_variance_like",
        "avg_topic_similarity",
        "umass_coherence",
        "silhouette_on_W",
        "sparsity_W",
        "sparsity_H",
        "stability",
    ]:
        plot_metric(metrics, col)

    return metrics

if test_best_k:
    # ---------- Example usage ----------
    # Assumes df2 with a 'paragraph' column.
    # Adjust K range and vectorizer settings to match corpus size.
    metrics_df = evaluate_nmf_k(
        df2['paragraph'].fillna("").astype(str),
        k_values=range(2, 31),
        ngram_range=(1, 1),
        min_df=2,
        max_df=0.95,
        max_features=None
    )
    print(metrics_df.round(4))


In [ ]:
# =========================
# Helpers
# =========================

def _find_topic_prob_cols(df: pd.DataFrame) -> List[str]:
    """
    Detect columns named like 'topic_<int>_prob' and return them
    sorted by their topic index.

    Example: ['topic_0_prob', 'topic_1_prob', ...]
    """
    cols = [c for c in df.columns if re.fullmatch(r"topic_\d+_prob", c)]
    cols = sorted(cols, key=lambda c: int(re.search(r"topic_(\d+)_prob", c).group(1)))
    return cols

def _ensure_save_dir(save_dir: Optional[str]) -> str:
    """
    Ensure there is a valid output directory and return its path.

    If `save_dir` is not provided:
      - Use '/kaggle/working' if it exists (Kaggle notebooks),
      - Otherwise fallback to '/mnt/data/output'.

    The directory is created if it does not exist.
    """
    if save_dir is None:
        if os.path.exists("/kaggle/working"):
            save_dir = "/kaggle/working"
        else:
            save_dir = "/mnt/data/output"
    os.makedirs(save_dir, exist_ok=True)
    return save_dir

def _sort_df(df: pd.DataFrame, sort_mode: str = "section_order0") -> pd.DataFrame:
    """
    Stable sorting of the DataFrame so the timeline is consistent.

    Modes
    -----
    - 'section_order0' (default): sort by section, then order0.
      As tie-breakers, if present, use 'folio' and 'line'.
    - 'order0'         : sort by order0 only (+ the same tie-breakers).

    Returns a new, stably-sorted DataFrame (mergesort).
    """
    sort_mode = (sort_mode or "section_order0").lower()
    if sort_mode == "order0":
        keys = ["order0"]
    else:
        keys = (["section"] if "section" in df.columns else []) + ["order0"]
    extras = [c for c in ["folio", "line"] if c in df.columns]
    return df.sort_values(keys + extras, kind="mergesort")  # stable sort

def _paragraph_weight_words(txt) -> float:
    """
    Paragraph weight function: number of words in the paragraph.

    Used to weight topic probabilities so longer paragraphs contribute more
    to tag-level aggregates.
    """
    if isinstance(txt, str):
        return float(len(txt.split()))
    return 0.0

def _stackplot(ax, x_values, y_matrix, labels, title, xlabel, title_pad=36, *, use_suptitle=True):
    """Stacked area with minimal config. Optionally use suptitle (for single-plot figs)."""
    ax.stackplot(x_values, y_matrix, labels=labels)
    ax.set_xlim(min(x_values), max(x_values))
    ax.set_ylim(0.0, 1.0)
    ax.set_ylabel("Topic %")
    ax.set_xlabel(xlabel)

    if use_suptitle:
        # Single-plot mode: use suptitle and adjust top margin
        fig = ax.figure
        ax.set_title("")
        fig.suptitle(title, y=0.82)
        fig.subplots_adjust(top=0.2)
    else:
        # Combined-figure mode: set a normal axes title, do NOT adjust the figure
        ax.set_title(title, pad=title_pad)

def _abbrev_section(name: Optional[str], n: int) -> str:
    """
    Abbreviate a section name to its first `n` alphanumeric characters in UPPERCASE.
    Falls back to 'UNK' for missing/blank names.
    """
    if not isinstance(name, str) or not name.strip():
        return "UNK"
    cleaned = "".join(ch for ch in name if ch.isalnum())
    if not cleaned:
        cleaned = name.strip()
    return cleaned[:n].upper()

def _draw_section_guides(ax, x_positions: List[int], section_ids: List[str],
                         top_pad: float = 0.06, abbrev_len: int = 3, fontsize: int = 12):
    """
    Draw vertical guide lines at section boundaries and label each
    contiguous section span at the top of the plot with an abbreviated name.

    Notes
    -----
    - Computes contiguous spans of identical section IDs along the x-axis.
    - Draws a vertical line at each boundary (midpoint between adjacent x positions).
    - Places rotated, bold section labels centered over each span.
    """
    if not x_positions:
        return

    # Identify contiguous spans for each section
    starts, ends, labels = [], [], []
    start_idx = 0
    for i in range(1, len(x_positions)):
        if section_ids[i] != section_ids[i-1]:
            starts.append(x_positions[start_idx])
            ends.append(x_positions[i-1])
            labels.append(section_ids[i-1])
            start_idx = i
    starts.append(x_positions[start_idx])
    ends.append(x_positions[-1])
    labels.append(section_ids[-1])

    # Vertical lines at boundaries
    for i in range(1, len(x_positions)):
        if section_ids[i] != section_ids[i-1]:
            cp = (x_positions[i-1] + x_positions[i]) / 2.0
            ax.axvline(cp, color="black", linewidth=2, zorder=5)

    # Abbreviated, vertical section labels centered over spans
    for s, e, lab in zip(starts, ends, labels):
        xc = (s + e) / 2.0
        txt = _abbrev_section(lab, abbrev_len)
        ax.text(xc, 1.1, txt, ha="center", va="top",
                rotation=90, rotation_mode="anchor",
                transform=ax.get_xaxis_transform(),
                fontsize=8, fontweight="bold")

def plot_topics_and_metrics(
    df: pd.DataFrame,
    K: int = 0,
    text_file: str | None = None,
    model_name: str | None = None,
    sort_mode: str = "section_order0",
    save_dir: Optional[str] = None,
    filename: Optional[str] = None,
    return_data: bool = False,
    abbrev_len: int = 3,
    section_label_fontsize: int = 16,
    *,
    plot_group_by: str = "folio",                # 'section' | 'folio' | 'language' | 'w_hand' | 'c_hand'
    metrics_df: Optional[pd.DataFrame] = None,
    metric_cols: Optional[list[str]] = None,
    normalize: str = "zscore",
) -> Tuple[str, pd.DataFrame]:
    group_map = {
        "folio": ("tag", "Folio"),
        "section": ("section", "Section"),
        "language": ("language", "Language"),
        "w_hand": ("writting_hand", "Writing hand"),
        "c_hand": ("currier_hand", "Currier hand"),
    }
    if plot_group_by not in group_map:
        raise ValueError(f"plot_group_by must be one of {list(group_map.keys())}")
    group_col, group_label = group_map[plot_group_by]

    if filename is None:
        filename = f"{K}_combined_stack_and_metrics.png"
    if text_file is not None:
        subfolder = f"{text_file}_{model_name or ''}".strip("_")
        save_dir = os.path.join("/kaggle/working/plots", subfolder)
    save_dir = _ensure_save_dir(save_dir)

    prob_cols = _find_topic_prob_cols(df)
    if not prob_cols:
        raise ValueError("No 'topic_#_prob' columns were found.")

    needed = {"order0", "paragraph", group_col}
    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    dfo = _sort_df(df, sort_mode).copy()
    weights = dfo["paragraph"].apply(_paragraph_weight_words)
    dfo["_w"] = weights

    # aggregate topic probabilities weighted by paragraph length
    weighted = dfo[prob_cols].multiply(weights, axis=0)
    grp_topic_sum = weighted.groupby(dfo[group_col]).sum()

    # normalize within each group
    denom = grp_topic_sum.sum(axis=1).replace(0.0, np.nan)
    grp_shares = grp_topic_sum.div(denom, axis=0).fillna(0.0)

    # earliest order per group
    grp_order0 = dfo.groupby(group_col)["order0"].min()

    # do we actually have 'section' available?
    has_section = "section" in dfo.columns

    # dominant section per group (guard all cases)
    if has_section:
        if group_col == "section":
            grp_section = pd.Series(index=grp_shares.index, data=grp_shares.index)
        else:
            sec_weight = dfo.pivot_table(
                index=group_col, columns="section", values="_w",
                aggfunc="sum", fill_value=0.0
            )
            grp_section = sec_weight.idxmax(axis=1)
    else:
        grp_section = pd.Series(index=grp_shares.index, data="Unknown")

    dfo = dfo.drop(columns=["_w"])

    meta = pd.DataFrame({
        group_col: grp_shares.index,
        f"{group_col}_order0": grp_order0.reindex(grp_shares.index).values,
        # this column will exist even if it's all "Unknown"
        "section": grp_section.reindex(grp_shares.index).values,
    }).set_index(group_col)

    # ordering
    if sort_mode == "order0":
        idx_ordered = meta.sort_values([f"{group_col}_order0"]).index
    else:
        # if we don't truly have section data, the values will be "Unknown" and this is harmless
        idx_ordered = meta.sort_values(["section", f"{group_col}_order0"]).index

    shares_ord = grp_shares.loc[idx_ordered]
    meta_ord = meta.loc[idx_ordered]

    # ---------- figure ----------
    import matplotlib.gridspec as gridspec
    fig = plt.figure(figsize=(14, 10))
    gs = gridspec.GridSpec(2, 1, height_ratios=[3, 2], hspace=0.25)

    # Top: stack plot
    ax1 = fig.add_subplot(gs[0, 0])
    x_vals = np.arange(1, len(shares_ord) + 1)
    y_matrix = shares_ord[prob_cols].T.values

    tf_label = text_file or ""
    mdl_label = model_name or ""
    title_left = f"{K} {mdl_label} {tf_label} Topics – Timeline by {group_label}".strip()

    _stackplot(
        ax1, x_vals, y_matrix, prob_cols,
        title=title_left,
        xlabel=group_label,
        title_pad=70,
        use_suptitle=False,
    )
    ax1.title.set_fontsize(ax1.title.get_fontsize() * 1.3)

    # Only draw guides if we truly have section info
    if has_section and "section" in meta_ord.columns:
        _draw_section_guides(
            ax1, x_vals.tolist(), meta_ord["section"].tolist(),
            top_pad=0.06, abbrev_len=abbrev_len, fontsize=section_label_fontsize
        )

    step = max(1, len(x_vals) // 25)
    tick_positions = x_vals[::step]
    tick_labels = [str(idx_ordered[i - 1]) for i in tick_positions]
    ax1.set_xticks(tick_positions)
    ax1.set_xticklabels(tick_labels, rotation=90, fontsize=8)

    # Bottom: metrics
    ax2 = fig.add_subplot(gs[1, 0])
    ax2.set_xlabel("K")
    ylab = {"zscore": "Z-scored metric (per model)",
            "minmax": "Normalized metric (min–max per model)",
            "raw": "Metric (raw scale)"}[normalize]
    ax2.set_ylabel(ylab)
    ax2.set_title("Metrics over K")

    if metrics_df is not None and len(metrics_df):
        mdf = metrics_df.copy()
        if "model" in mdf.columns and model_name is not None:
            mdf = mdf[mdf["model"] == model_name]
        if "K" in mdf.columns:
            mdf["K"] = pd.to_numeric(mdf["K"], errors="coerce")
            mdf = mdf[mdf["K"] <= K].sort_values("K")
        if "K" in mdf.columns and len(mdf):
            ks = sorted(pd.unique(mdf["K"].astype(int)))
            ax2.set_xticks(ks)

        if metric_cols is None:
            metric_cols = [
                c for c in mdf.columns
                if c not in {"model", "K"} and np.issubdtype(mdf[c].dtype, np.number)
            ]

        lower_is_better = {"avg_topic_similarity", "perplexity", "reconstruction_error"}
        plot_df = mdf.copy()
        for col in metric_cols:
            if col in plot_df.columns:
                plot_df[col] = plot_df[col].astype(float)
                if col in lower_is_better:
                    plot_df[col] = -plot_df[col]

        rescale_by_first = {"stability", "perplexity", "reconstruction_error"}
        for col in rescale_by_first:
            if col in plot_df.columns and len(plot_df):
                first_val = plot_df[col].iloc[0]
                if np.isfinite(first_val) and first_val != 0:
                    plot_df[col] = 1 - plot_df[col] - abs(first_val)

        def _zscore(x):
            m, s = np.nanmean(x), np.nanstd(x)
            return (x - m) / s if s > 0 else np.zeros_like(x)

        def _minmax(x):
            xmin, xmax = np.nanmin(x), np.nanmax(x)
            rng = xmax - xmin
            return (x - xmin) / rng if rng > 0 else np.zeros_like(x)

        for col in metric_cols:
            if col not in plot_df.columns:
                continue
            vals = plot_df[col].values
            if np.all(np.isnan(vals)):
                continue
            if col in rescale_by_first or normalize == "raw":
                yvals = vals
            elif normalize == "zscore":
                yvals = _zscore(vals)
            else:
                yvals = _minmax(vals)
            ax2.plot(plot_df["K"].values, yvals, marker="o", label=col)

        ax2.grid(True, alpha=0.3)
        ax2.legend(loc="best")

    fig.tight_layout(rect=[0, 0, 1, 0.97])

    out_path = os.path.join(save_dir, filename)
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()

    # ---- compact output table ----
    out_df = shares_ord.copy()

    # Collect meta columns to prepend (avoid name collision if grouping by 'section')
    meta_cols = {}
    if has_section and "section" in meta_ord.columns:
        meta_section_name = "dominant_section" if group_col == "section" else "section"
        meta_cols[meta_section_name] = meta_ord["section"]
    meta_cols[f"{group_col}_order0"] = meta_ord[f"{group_col}_order0"]

    # Prepend meta columns in order (reverse insert)
    for colname, series in reversed(list(meta_cols.items())):
        out_df.insert(0, colname, series)

    # Bring index (group key) out as a column without colliding
    out_df.reset_index(inplace=True)               # index becomes a column named `group_col`
    out_df.rename(columns={group_col: "group"}, inplace=True)

    # If we grouped by section and there's no plain 'section' column yet, add it from the group key
    if group_col == "section" and "section" not in out_df.columns:
        out_df.insert(0, "section", out_df["group"])

    return out_path, (out_df if return_data else out_df.head(10))



def plot_topics_by_paragraph_with_sections(
    df: pd.DataFrame,
    K: int = 0,
    text_file: str | None = None,
    model_name: str | None = None,
    sort_mode: str = "section_order0",
    save_dir: Optional[str] = None,
    filename: Optional[str] = None,
    return_data: bool = False,
    abbrev_len: int = 3,
    section_label_fontsize: int = 16,
    *,
    metrics_df: Optional[pd.DataFrame] = None,     # history across Ks (all models)
    metric_cols: Optional[list[str]] = None,       # e.g. ['coherence_umass','stability','avg_topic_similarity','perplexity','reconstruction_error']
    normalize: str = "raw",                        # 'raw' | 'zscore' | 'minmax'
    rescale_by_first: set[str] = frozenset({"stability"}),  # metrics to divide by their first value
) -> Tuple[str, pd.DataFrame]:
    """
    Top: paragraph-level stacked area of topic probabilities (row-normalized).
    Bottom: line plot with selected metrics vs K for this model (<= current K),
            with options to invert 'lower is better', rescale-by-first, and normalize.
    """
    if filename is None:
        filename = f"{K}_topic_stack_by_paragraph_sections_combined.png"

    if text_file is not None:
        subfolder = f"{text_file}_{model_name or ''}".strip("_")
        save_dir = os.path.join("/kaggle/working/plots", subfolder)
    save_dir = _ensure_save_dir(save_dir)

    # ---- data checks ----
    prob_cols = _find_topic_prob_cols(df)
    if not prob_cols:
        raise ValueError("No 'topic_#_prob' columns were found.")

    dfo = _sort_df(df.copy(), sort_mode)

    # ---- row-normalize per paragraph ----
    probs = dfo[prob_cols].copy()
    row_sums = probs.sum(axis=1).replace(0.0, np.nan)
    probs = probs.div(row_sums, axis=0).fillna(0.0)

    x = np.arange(1, len(probs) + 1)
    y_matrix = probs[prob_cols].T.values

    # ---- figure (top=stack, bottom=metrics) ----
    import matplotlib.gridspec as gridspec
    fig = plt.figure(figsize=(14, 10))
    gs = gridspec.GridSpec(2, 1, height_ratios=[3, 2], hspace=0.25)

    # Top: paragraph stack
    ax1 = fig.add_subplot(gs[0, 0])
    tf_label = text_file or ""
    mdl_label = model_name or ""
    title_top = f"{K} {mdl_label} {tf_label} Topics – Timeline by paragraph".strip()

    _stackplot(
        ax1, x, y_matrix, prob_cols,
        title=title_top,
        xlabel="Paragraph",
        title_pad=70,
        use_suptitle=False,   # IMPORTANT in combined layout
    )
    ax1.title.set_fontsize(ax1.title.get_fontsize() * 1.3)

    # Section guides (if available)
    sections = dfo["section"].tolist() if "section" in dfo.columns else ["Unknown"] * len(dfo)
    _draw_section_guides(
        ax1, x.tolist(), sections,
        top_pad=0.06, abbrev_len=abbrev_len, fontsize=section_label_fontsize
    )

    step = max(1, len(x) // 25)
    ax1.set_xticks(x[::step])

    # Bottom: metrics
    ax2 = fig.add_subplot(gs[1, 0])
    ax2.set_xlabel("K")
    ylab = {
        "raw": "Metric (raw / rescaled-by-first)",
        "zscore": "Z-scored metric (per model)",
        "minmax": "Normalized metric (min–max per model)",
    }[normalize]
    ax2.set_ylabel(ylab)
    ax2.set_title("Metrics over K")

    if metrics_df is not None and len(metrics_df):
        mdf = metrics_df.copy()

        # Filter by model and K<=current
        if "model" in mdf.columns and model_name is not None:
            mdf = mdf[mdf["model"] == model_name]
        if "K" in mdf.columns:
            mdf["K"] = pd.to_numeric(mdf["K"], errors="coerce")
            mdf = mdf[mdf["K"] <= K].sort_values("K")

        if "K" in mdf.columns and len(mdf):
            ks = sorted(pd.unique(mdf["K"].astype(int)))
            ax2.set_xticks(ks)

        # Default: all numeric metrics except K/model
        if metric_cols is None:
            metric_cols = [
                c for c in mdf.columns
                if c not in {"model", "K"} and np.issubdtype(mdf[c].dtype, np.number)
            ]

        # Invert where lower is better so "up = good"
        lower_is_better = {"avg_topic_similarity", "perplexity", "reconstruction_error"}
        plot_df = mdf.copy()
        for col in metric_cols:
            if col in plot_df.columns:
                plot_df[col] = plot_df[col].astype(float)
                if col in lower_is_better:
                    plot_df[col] = -plot_df[col]

        # Rescale-by-first (AFTER inversion)
        for col in rescale_by_first:
            if col in plot_df.columns and len(plot_df):
                first_val = plot_df[col].iloc[0]
                if np.isfinite(first_val) and first_val != 0:
                    plot_df[col] = plot_df[col] / abs(first_val)

        # Normalization mode (apply only to series NOT rescaled-by-first)
        def _zscore(x):
            m, s = np.nanmean(x), np.nanstd(x)
            return (x - m) / s if s > 0 else np.zeros_like(x)

        def _minmax(x):
            xmin, xmax = np.nanmin(x), np.nanmax(x)
            rng = xmax - xmin
            return (x - xmin) / rng if rng > 0 else np.zeros_like(x)

        for col in metric_cols:
            if col not in plot_df.columns:
                continue
            vals = plot_df[col].values
            if np.all(np.isnan(vals)):
                continue

            if col in rescale_by_first or normalize == "raw":
                yvals = vals
            elif normalize == "zscore":
                yvals = _zscore(vals)
            else:
                yvals = _minmax(vals)

            ax2.plot(plot_df["K"].values, yvals, marker="o", label=col)

        ax2.grid(True, alpha=0.3)
        ax2.legend(loc="best")

    fig.tight_layout(rect=[0, 0, 1, 0.97])

    out_path = os.path.join(save_dir, filename)
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()

    # Return original rows + normalized probs
    out_df = dfo.reset_index(drop=True).copy()
    out_df = pd.concat([out_df, probs.reset_index(drop=True)], axis=1)
    return out_path, (out_df if return_data else out_df.head(10))


In [ ]:
def _natural_key(s: str):
    """
    Produce a key for "natural" sorting of filenames like: 1, 2, 10 (instead of 1, 10, 2).
    Splits the string into digit and non-digit chunks and casts digit chunks to int.
    """
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", s)]

def make_gif_from_pngs(
    input_glob: str,
    out_path: str,
    hold_ms: int = 3000,      # 3 seconds per still frame
    fade_ms: int = 800,       # crossfade duration between frames
    fade_steps: int = 8,      # number of intermediate frames for the fade
    quant_colors: int = 128,  # palette size: 64=smaller file, 256=higher quality
):
    """
    Build an animated GIF from a set of PNGs matched by a glob pattern.
    The routine:
      - loads all images and downsizes them to 50% (for smaller GIFs),
      - inserts a 'hold' of `hold_ms` on each image,
      - adds a crossfade between consecutive images using `fade_steps`,
      - quantizes frames to a reduced palette to compress the output,
      - saves an optimized looping GIF.

    Parameters
    ----------
    input_glob : str
        Glob pattern pointing to the PNG frames, e.g. '/.../sub/*.png'.
        Files are sorted with natural ordering so '2' < '10'.
    out_path : str
        Output GIF path. Parent directories are created if needed.
    hold_ms : int
        Duration (milliseconds) to show each image as a still frame.
    fade_ms : int
        Total duration (milliseconds) of the crossfade between consecutive images.
    fade_steps : int
        Number of intermediate blended frames inserted during the crossfade.
        More steps = smoother fade but larger file.
    quant_colors : int
        Target number of colors for GIF palette quantization (tradeoff size/quality).

    Returns
    -------
    str
        The path to the saved GIF.

    Raises
    ------
    ValueError
        If fewer than 2 input images are found.
    """
    paths = sorted(glob.glob(input_glob), key=_natural_key)
    if len(paths) < 2:
        raise ValueError(f"Need at least 2 images to create a GIF with fades. Empty/insufficient glob: {input_glob}")

    # Load images as RGBA and downscale to 50% using high-quality resampling
    imgs_rgba: List[Image.Image] = []
    for p in paths:
        im = Image.open(p).convert("RGBA")
        w, h = im.size
        im_small = im.resize((max(1, w // 2), max(1, h // 2)), Image.LANCZOS)
        imgs_rgba.append(im_small)

    frames_p: List[Image.Image] = []
    durations: List[int] = []

    def _to_P(img: Image.Image) -> Image.Image:
        """
        Convert to an 8-bit palette image to reduce file size.
        Uses median-cut (ADAPTIVE) quantization with a capped color count.
        """
        return img.convert("RGB").quantize(colors=quant_colors, method=Image.MEDIANCUT, kmeans=0)

    # First still frame
    frames_p.append(_to_P(imgs_rgba[0]))
    durations.append(hold_ms)

    # Crossfades + stills for subsequent images
    for i in range(len(imgs_rgba) - 1):
        a = imgs_rgba[i]
        b = imgs_rgba[i + 1]

        # Insert intermediate blended frames for a smooth crossfade
        if fade_steps > 0 and fade_ms > 0:
            step_ms = max(1, fade_ms // fade_steps)
            for k in range(1, fade_steps + 1):
                # alpha in (0, 1) but not including endpoints to avoid duplicates
                alpha = k / (fade_steps + 1)
                blended = Image.blend(a, b, alpha)
                frames_p.append(_to_P(blended))
                durations.append(step_ms)

        # Then hold on the next full image
        frames_p.append(_to_P(b))
        durations.append(hold_ms)

    # Save an optimized, looping GIF
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    first, rest = frames_p[0], frames_p[1:]
    first.save(
        out_path,
        save_all=True,
        append_images=rest,
        duration=durations,
        loop=0,
        optimize=True,  # let Pillow try to compress the GIF
        disposal=2,     # restore to background before drawing next frame (helps avoid artifacts)
    )
    return out_path

def make_gifs_from_subfolders(
    root_dir: str = "/kaggle/working/plots",
    pattern: str = "*.png",
    out_dir: Optional[str] = None,
    hold_ms: int = 3000,
    fade_ms: int = 800,
    fade_steps: int = 8,
    quant_colors: int = 128,
) -> List[str]:
    """
    Iterate over immediate subfolders of `root_dir` and create one GIF per subfolder.

    For each subfolder:
      - Collect input frames using `pattern` (e.g., '*.png'), natural-sorted.
      - Build a GIF using `make_gif_from_pngs`.
      - Save it either inside that subfolder as '<subfolder>.gif' or into `out_dir`.

    Parameters
    ----------
    root_dir : str
        Directory that contains subfolders, each with PNGs to convert.
    pattern : str
        Glob to select frames inside each subfolder (default: '*.png').
    out_dir : str or None
        If provided, all GIFs are saved here (created if missing).
        Otherwise, each GIF is saved next to its frames in the subfolder.
    hold_ms, fade_ms, fade_steps, quant_colors :
        Passed through to `make_gif_from_pngs` to control timing and compression.

    Returns
    -------
    List[str]
        A list of full paths to the created GIFs.

    Raises
    ------
    ValueError
        If no subfolders are found under `root_dir`.
    RuntimeError
        If no GIFs were created (e.g., no frames matched in any subfolder).
    """
    # Enumerate immediate subfolders
    subfolders = sorted(
        [d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))],
        key=_natural_key
    )
    if not subfolders:
        raise ValueError(f"No subfolders found under: {root_dir}")

    created = []
    for sub in subfolders:
        sub_path = os.path.join(root_dir, sub)
        input_glob = os.path.join(sub_path, pattern)

        # Decide output location
        if out_dir:
            os.makedirs(out_dir, exist_ok=True)
            out_path = os.path.join(out_dir, f"{sub}.gif")
        else:
            out_path = os.path.join(sub_path, f"{sub}.gif")

        try:
            gif_path = make_gif_from_pngs(
                input_glob=input_glob,
                out_path=out_path,
                hold_ms=hold_ms,
                fade_ms=fade_ms,
                fade_steps=fade_steps,
                quant_colors=quant_colors,
            )
            created.append(gif_path)
            print(f"[OK] {gif_path}")
        except Exception as e:
            # Skip subfolders that fail (e.g., not enough frames) but continue with others
            print(f"[SKIP] {sub_path}: {e}")

    if not created:
        raise RuntimeError("No GIFs were created. Check your patterns or folder contents.")
    return created


In [ ]:
# NOTE:
# - The functions below expect that you already have:
#     * df2: a DataFrame holding your metadata + 'paragraph' column (used for indices/concat)
#     * vectorizer: a fitted TfidfVectorizer (used only to read feature names or config)
#   If you want the functions to be self-contained, pass those objects in as parameters.

def train_NMF(
    X,
    K: int,
    *,
    random_state: int = 0,
    topn: int = 10,
) -> tuple[pd.DataFrame, dict[str, list], SKNMF]:
    """
    Train NMF on a precomputed TF-IDF matrix X and return:
      - df_topics with topic_<i>_prob and topic_assigned
      - topic_terms dict
      - fitted NMF model

    Relies on globals: df2 (rows align with X) and vectorizer (for feature names).
    """
    # Safety: align lengths
    if X.shape[0] != len(df2):
        raise ValueError(f"X has {X.shape[0]} rows but df2 has {len(df2)}.")

    nmf = SKNMF(
        n_components=K,
        init="nndsvda",
        random_state=random_state,
        max_iter=500,
        solver="cd",
        beta_loss="frobenius",
        tol=1e-4,
    )
    W = nmf.fit_transform(X)               # (n_docs, K)
    H = nmf.components_                    # (K, n_terms)

    # Probabilities per document
    W_prob = normalize(W, norm="l1", axis=1)
    prob_cols = [f"topic_{i}_prob" for i in range(K)]
    df_probs = pd.DataFrame(W_prob, columns=prob_cols, index=df2.index)

    # Hard assignment
    assigned = np.argmax(W_prob, axis=1)
    df_topics = df2.copy()
    df_topics["topic_assigned"] = assigned
    df_topics = pd.concat([df_topics, df_probs], axis=1)

    # Top terms per topic
    feature_names = np.array(vectorizer.get_feature_names_out())
    topn = min(topn, H.shape[1])
    topic_terms = {
        f"topic_{i}": feature_names[np.argsort(H[i])[::-1][:topn]].tolist()
        for i in range(K)
    }

    return df_topics, topic_terms, nmf


def train_LDA(texts, K: int) -> Tuple[pd.DataFrame, Dict[str, list], LatentDirichletAllocation, CountVectorizer]:
    """
    Train an LDA topic model (scikit-learn) on raw texts using a CountVectorizer
    configured similarly to your TF-IDF vectorizer.

    Returns:
      - df_topics: df2 + topic_<i>_prob cols + topic_assigned
      - topic_terms: top words per actual topic
      - lda: fitted model
      - cv: CountVectorizer used for the counts
    """
    # 1) CountVectorizer mirroring your TF-IDF settings where sensible
    cv = CountVectorizer(
        ngram_range=getattr(vectorizer, "ngram_range", (1,1)),
        min_df=getattr(vectorizer, "min_df", 1),
        max_df=getattr(vectorizer, "max_df", 1.0),
        strip_accents=getattr(vectorizer, "strip_accents", None),
        lowercase=getattr(vectorizer, "lowercase", True),
        token_pattern=getattr(vectorizer, "token_pattern", r"(?u)\b\w+\b"),
        vocabulary=None
    )
    Xc = cv.fit_transform(texts)
    feature_names = np.array(cv.get_feature_names_out())

    # 2) Fit LDA
    lda = LatentDirichletAllocation(
        n_components=K,
        max_iter=1000,
        learning_method="batch",
        random_state=42,
        evaluate_every=0,
        doc_topic_prior=None,
        topic_word_prior=None
    )
    W = lda.fit_transform(Xc)                 # (n_docs, n_topics_effective)
    W_prob = normalize(W, norm="l1", axis=1)  # row-normalized

    # Use *actual* numbers from the model outputs
    n_topics_W = W_prob.shape[1]
    H = lda.components_                       # (n_topics_effective, n_terms)
    n_topics_H = H.shape[0]

    # Safety: ensure both views agree, otherwise take the minimum
    n_topics = min(n_topics_W, n_topics_H)
    if n_topics != K:
        print(f"[LDA] Requested K={K}, effective topics={n_topics} "
              f"(W has {n_topics_W}, H has {n_topics_H}).")

    # 3) Probabilities & hard assignments (truncate if needed)
    prob_cols = [f"topic_{i}_prob" for i in range(n_topics)]
    W_prob_use = W_prob[:, :n_topics]        # align to n_topics
    df_probs = pd.DataFrame(W_prob_use, columns=prob_cols, index=df2.index)

    assigned = np.argmax(W_prob_use, axis=1)
    df_topics = df2.copy()
    df_topics["topic_assigned"] = assigned
    df_topics = pd.concat([df_topics, df_probs], axis=1)

    # 4) Top terms per topic
    topn = min(max(1, K), H.shape[1])        # cap by vocab size
    topic_terms: Dict[str, list] = {
        f"topic_{i}": feature_names[np.argsort(H[i])[::-1][:topn]].tolist()
        for i in range(n_topics)
    }

    return df_topics, topic_terms, lda, cv


def train_BERTopic(
    texts,
    K: int,
    embedding_model: str = "all-MiniLM-L6-v2",
    *,
    random_state: int = 0,
    min_topic_size: int = 10,
    return_feature_names: bool = False,
) -> Tuple[pd.DataFrame, Dict[str, list], BERTopic, CountVectorizer, List[int], np.ndarray | None]:
    """
    Train BERTopic and reduce to exactly K topics. Returns:
      - df_topics (with topic_<i>_prob and topic_assigned in reindexed 0..K-1 space)
      - topic_terms dict
      - model (BERTopic)
      - cv (CountVectorizer used internally)
      - topic_ids (BERTopic's internal topic ids in the order you used)
      - feature_names (np.ndarray) if return_feature_names=True, else None

    Notes
    -----
    - For reproducibility, we pass a random_state. For *full* control, also pass
      custom UMAP/HDBSCAN with fixed seeds (see commented lines).
    """
    # cv tuned to avoid min_df/max_df issues when topics are few
    cv = CountVectorizer(
        ngram_range=getattr(vectorizer, "ngram_range", (1,1)),
        min_df=1,
        max_df=1.0,
        strip_accents=getattr(vectorizer, "strip_accents", None),
        lowercase=getattr(vectorizer, "lowercase", True),
        token_pattern=getattr(vectorizer, "token_pattern", r"(?u)\b\w+\b"),
    )

    # Optional: deterministic UMAP/HDBSCAN (uncomment if you want stricter reproducibility)
    # umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, random_state=random_state)
    # hdbscan_model = HDBSCAN(min_cluster_size=min_topic_size, prediction_data=True)
    # model = BERTopic(embedding_model=embedding_model, vectorizer_model=cv,
    #                  umap_model=umap_model, hdbscan_model=hdbscan_model,
    #                  calculate_probabilities=True, nr_topics=K, verbose=False)

    model = BERTopic(
        embedding_model=embedding_model,
        vectorizer_model=cv,
        calculate_probabilities=True,
        nr_topics=K,
        verbose=False,
        min_topic_size=min_topic_size,
    )

    topics, probs = model.fit_transform(texts)

    # Order topics by frequency (exclude -1) and reorder probabilities
    info = model.get_topic_info()
    info = info[info["Topic"] != -1].sort_values("Count", ascending=False).reset_index(drop=True)
    topic_ids = info["Topic"].tolist()

    topic_indices_sorted = sorted([t for t in model.get_topics().keys() if t != -1])
    oldid2col = {tid: i for i, tid in enumerate(topic_indices_sorted)}
    cols_in_new_order = [oldid2col[tid] for tid in topic_ids]
    probs_reordered = probs[:, cols_in_new_order]

    probs_norm = normalize(probs_reordered, norm="l1", axis=1)
    assigned_new = probs_norm.argmax(axis=1)

    prob_cols = [f"topic_{i}_prob" for i in range(len(topic_ids))]
    df_probs = pd.DataFrame(probs_norm, columns=prob_cols, index=df2.index)

    df_topics = df2.copy()
    df_topics["topic_assigned"] = assigned_new
    df_topics = pd.concat([df_topics, df_probs], axis=1)

    # Top words per reindexed topic
    topn = K
    topic_terms: Dict[str, list] = {}
    for new_i, tid in enumerate(topic_ids):
        words_scores = model.get_topic(tid) or []
        topic_terms[f"topic_{new_i}"] = [w for (w, s) in words_scores[:topn]]

    feature_names = np.array(cv.get_feature_names_out()) if return_feature_names else None
    return df_topics, topic_terms, model, cv, topic_ids, feature_names

In [ ]:
# ---------- 1) Vectorization ----------
# Build TF-IDF from paragraphs once. NMF uses this matrix directly.
# LDA/BERTopic will use their own count-based/vectorizer pipelines internally.
texts = df2['paragraph'].fillna("").astype(str)

vectorizer = TfidfVectorizer(
    ngram_range=(1, 1),
    min_df=2,
    max_df=0.95,
    strip_accents=None,
    lowercase=False,            # Voynich often benefits from preserving case/diacritics
    token_pattern=r"(?u)\b\w+\b"
)
X = vectorizer.fit_transform(texts)

# ---------- 2) Train models across methods and K, then plot ----------
# We iterate over modeling methods and topic counts. For each run, we:
#   - train the model (getting per-document topic probabilities in df_topics),
#   - plot a TAG-level stacked timeline with section guides,
#   - (optionally) you can also plot the paragraph-level timeline (commented below).
metrics_by_model = {m: [] for m in ['NMF', 'BerTOPIC', 'LDA']}

# All metrics we might plot (some will be NaN depending on the model)
metric_cols = [
    "coherence_umass",
    "stability",
    "avg_topic_similarity",
    "perplexity",             # LDA only
    "reconstruction_error",   # NMF only
    "reconstruction_like",    # placeholder for BERTopic (NaN)
]

def _concat_metrics(metrics_by_model):
    frames = []
    for m, rows in metrics_by_model.items():
        if rows:
            dfm = pd.DataFrame(rows)
            if "model" not in dfm.columns:
                dfm["model"] = m
            frames.append(dfm)
    if frames:
        # ensure K is numeric for sorting
        out = pd.concat(frames, ignore_index=True)
        if "K" in out.columns:
            out["K"] = pd.to_numeric(out["K"], errors="coerce")
        return out
    # empty scaffold so plotting code doesn't break the first time
    return pd.DataFrame(columns=["model", "K"] + metric_cols)

for model_name in ['NMF', 'BerTOPIC', 'LDA']:
    for K in range(min_K, max_K + 1):

        if model_name == "NMF":
            df_topics, topic_terms, nmf = train_NMF(X, K)
            H = nmf.components_
            Hn = normalize(H, norm="l1", axis=1)
            X_bin = (X > 0).astype(int)

            coh = umass_coherence(Hn, X_bin, top_n=15)
            avg_sim = avg_topic_similarity(Hn)

            nmf2 = SKNMF(n_components=K, init="nndsvda", random_state=42, max_iter=500,
                         solver="cd", beta_loss="frobenius", tol=1e-4)
            _ = nmf2.fit_transform(X)
            H2n = normalize(nmf2.components_, norm="l1", axis=1)
            stab = topic_stability(Hn, H2n)

            fit_quality = nmf.reconstruction_err_
            fit_name = "reconstruction_error"

        elif model_name == "BerTOPIC":
            df_topics, topic_terms, model_bt, cv_bt, topic_ids, feature_names = train_BERTopic(
                texts, K, embedding_model="all-MiniLM-L6-v2",
                random_state=0, min_topic_size=10, return_feature_names=False
            )

            tid2row = {tid: i for i, tid in enumerate(sorted(t for t in model_bt.get_topics().keys() if t != -1))}
            rows = [tid2row[tid] for tid in topic_ids]
            H = model_bt.c_tf_idf_[rows, :]
            Hn = normalize(H, norm="l1", axis=1)

            Xc_bt = cv_bt.transform(texts)
            X_bin = (Xc_bt > 0).astype(int)

            coh = umass_coherence(Hn, X_bin, top_n=15)
            avg_sim = avg_topic_similarity(Hn)

            # stability
            _, _, model_bt2, cv_bt2, topic_ids2, _ = train_BERTopic(
                texts, K, embedding_model="all-MiniLM-L6-v2",
                random_state=42, min_topic_size=10, return_feature_names=False
            )
            tid2row2 = {tid: i for i, tid in enumerate(sorted(t for t in model_bt2.get_topics().keys() if t != -1))}
            rows2 = [tid2row2[tid] for tid in topic_ids2]
            H2 = model_bt2.c_tf_idf_[rows2, :]
            H2n = normalize(H2, norm="l1", axis=1)
            stab = topic_stability(Hn, H2n)

            fit_quality = np.nan
            fit_name = "reconstruction_like"

        elif model_name == "LDA":
            # IMPORTANT: keep the LDA CountVectorizer separate
            df_topics, topic_terms, lda, cv_lda = train_LDA(texts, K)

            H = lda.components_
            Hn = normalize(H, norm="l1", axis=1)

            # Build X_bin with the *same* cv the LDA was fit with
            Xc_lda = cv_lda.transform(texts)
            X_bin = (Xc_lda > 0).astype(int)

            coh = umass_coherence(Hn, X_bin, top_n=15)
            avg_sim = avg_topic_similarity(Hn)

            lda2 = LatentDirichletAllocation(
                n_components=K, max_iter=lda.max_iter, learning_method=lda.learning_method,
                random_state=42, evaluate_every=0, doc_topic_prior=lda.doc_topic_prior,
                topic_word_prior=lda.topic_word_prior
            ).fit(Xc_lda)
            H2n = normalize(lda2.components_, norm="l1", axis=1)
            stab = topic_stability(Hn, H2n)

            fit_quality = lda.perplexity(Xc_lda)
            fit_name = "perplexity"

        elif model_name == "BerTOPIC":
            df_topics, topic_terms, model, cv, topic_ids, feature_names = train_BERTopic(
                texts, K, embedding_model="all-MiniLM-L6-v2",
                random_state=0, min_topic_size=10, return_feature_names=False
            )

            # --- Build H from c-TF-IDF in the same topic order (you already have this) ---
            tid2row = {tid: i for i, tid in enumerate(sorted(t for t in model.get_topics().keys() if t != -1))}
            rows = [tid2row[tid] for tid in topic_ids]
            H = model.c_tf_idf_[rows, :]
            Hn = normalize(H, norm="l1", axis=1)
            
            # --- IMPORTANT: build X_bin using the SAME vocabulary/processing as c_tf_idf_ ---
            vec = model.vectorizer_model  # this is the fitted CountVectorizer used by BERTopic on topic-docs
            
            # Get feature names in the exact column order of c_tf_idf_
            try:
                feature_names_bt = np.array(vec.get_feature_names_out())
            except Exception:
                # Fallback if get_feature_names_out is unavailable
                vocab = vec.vocabulary_
                feature_names_bt = np.array(sorted(vocab, key=vocab.get))
            
            # Recreate a CountVectorizer with the same analyzer settings but FIXED vocabulary
            cv_align = CountVectorizer(
                vocabulary=feature_names_bt,
                ngram_range=getattr(vec, "ngram_range", (1, 1)),
                lowercase=getattr(vec, "lowercase", True),
                strip_accents=getattr(vec, "strip_accents", None),
                token_pattern=getattr(vec, "token_pattern", r"(?u)\b\w+\b"),
                analyzer=getattr(vec, "analyzer", "word"),
                # min_df/max_df are ignored when vocabulary is fixed (fine, we need alignment not pruning)
            )
            
            Xc_align = cv_align.transform(texts)         # counts on the SAME vocab as c_tf_idf_
            X_bin = (Xc_align > 0).astype(int)           # binary doc-term
            
            # Now shapes match: H.shape[1] == X_bin.shape[1]
            coh = umass_coherence(Hn, X_bin, top_n=15)
            avg_sim = avg_topic_similarity(Hn)

            # Stability via refit
            df_topics2, topic_terms2, model2, cv2, topic_ids2, _ = train_BERTopic(
                texts, K, embedding_model="all-MiniLM-L6-v2",
                random_state=42, min_topic_size=10, return_feature_names=False
            )
            tid2row2 = {tid: i for i, tid in enumerate(sorted(t for t in model2.get_topics().keys() if t != -1))}
            rows2 = [tid2row2[tid] for tid in topic_ids2]
            H2 = model2.c_tf_idf_[rows2, :]
            H2n = normalize(H2, norm="l1", axis=1)
            stab = topic_stability(Hn, H2n)

            fit_quality = np.nan
            fit_name = "reconstruction_like"

        # ---- append metrics row ----
        metrics_row = {
            "model": model_name,
            "K": K,
            "coherence_umass": float(coh),
            "avg_topic_similarity": float(avg_sim),  # lower is better
            "stability": float(stab),
            fit_name: float(fit_quality) if not (isinstance(fit_quality, float) and np.isnan(fit_quality)) else np.nan,
        }
        metrics_by_model[model_name].append(metrics_row)

        # ---- build up-to-date metrics_df_all and plot combined figure ----
        metrics_df_all = _concat_metrics(metrics_by_model)

        if plot_by_folio:
            plot_topics_and_metrics(
                df_topics,
                K=K,
                text_file=text_to_read,
                model_name=model_name,
                plot_group_by=plot_group_by,
                sort_mode="section_order0",
                save_dir=None,
                abbrev_len=4,
                section_label_fontsize=16,
                metrics_df=metrics_df_all,      # now defined & current
                metric_cols=metric_cols,        # list of series to try plotting
            )

        if plot_by_paragraph:
            # ---- OPTIONAL: Paragraph-level stacked timeline with sections ----
            # Useful to inspect fine-grained topic variation, but can be very long.
            plot_topics_by_paragraph_with_sections(
                df_topics,
                K=K,
                text_file=text_to_read,
                model_name=model_name,
                plot_group_by=plot_group_by,
                sort_mode="section_order0",
                save_dir=None,
                abbrev_len=4,
                section_label_fontsize=16
            )
            
# ---------- 3) Build GIFs from all subfolders ----------
# After running all K/model combinations, each subfolder under /kaggle/working/plots
# should contain PNGs. This step creates one GIF per subfolder and stores them centrally.
make_gifs_from_subfolders(
    "/kaggle/working/plots",
    pattern="*.png",
    out_dir="/kaggle/working/plots_gifs",
    hold_ms=3000,  # 3s per still image
    fade_ms=800,   # crossfade duration
    fade_steps=8,  # number of intermediate frames in the fade
    quant_colors=128
)
